In [3]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append("../")
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import ogbench

from utils.datasets import GCDataset, Dataset, HGCDataset
from utils.flax_utils import restore_agent
from agents.ota.original import OTAAgent, get_config as get_config_ota

In [ ]:
env_cube_single, train_dataset_cube, _ = ogbench.make_env_and_datasets(
    dataset_name="cube-single-play-v0"
)
train_dataset_cube = GCDataset(
    Dataset.create(norm=False, **train_dataset_cube), agent_config
)
example_batch_cube = train_dataset_cube.sample(1)
agent_config = get_config_ota()
ota_agent = OTAAgent.create(
    0,
    example_batch_cube["observations"],
    example_batch_cube["actions"],
    agent_config,
    ex_goals=None,
) 

# agent_gcivl = restore_agent(
#     agent_gcivl,
#     "../results/gcivl/antmaze-large-navigate-v0_gcivl_sd000/checkpoints",
#     step=1_000_000,
# )



## Visualizing expert trajectories for Cube-single

In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
#  Cube-Single Task Trajectory Visualiser
# ═══════════════════════════════════════════════════════════════════════════════

# ── Headless rendering fix – MUST be set before any mujoco import ─────────────
import os
os.environ["MUJOCO_GL"] = "egl"        # GPU-accelerated headless (preferred)
# os.environ["MUJOCO_GL"] = "osmesa"   # pure-software fallback (no GPU needed)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.animation as animation
from IPython.display import HTML, display
from PIL import Image, ImageDraw

import mujoco
import gymnasium as gym
import ogbench
import ogbench.manipspace


# ─── CONSTANTS ────────────────────────────────────────────────────────────────
XYZ_CENTER    = np.array([0.425, 0.0, 0.0])
XYZ_SCALE     = 10.0
GRIPPER_SCALE = 3.0
CUBE_SIZE_M   = 0.04

OBJ_X_RANGE = (0.30, 0.55)
OBJ_Y_RANGE = (-0.30, 0.30)

CUBE_SINGLE_TASKS = {
    1: dict(name="horizontal",  init_xyz=np.array([0.425,  0.1,  0.02]), goal_xyz=np.array([0.425, -0.1,  0.02])),
    2: dict(name="vertical1",   init_xyz=np.array([0.350,  0.0,  0.02]), goal_xyz=np.array([0.500,  0.0,  0.02])),
    3: dict(name="vertical2",   init_xyz=np.array([0.500,  0.0,  0.02]), goal_xyz=np.array([0.350,  0.0,  0.02])),
    4: dict(name="diagonal1",   init_xyz=np.array([0.350, -0.2,  0.02]), goal_xyz=np.array([0.500,  0.2,  0.02])),
    5: dict(name="diagonal2",   init_xyz=np.array([0.350,  0.2,  0.02]), goal_xyz=np.array([0.500, -0.2,  0.02])),
}


# ─── OBSERVATION DECODING ─────────────────────────────────────────────────────
def decode_effector_pos(obs):
    return obs[..., 12:15] / XYZ_SCALE + XYZ_CENTER

def decode_block_pos(obs, block_idx=0):
    s = 19 + block_idx * 9
    return obs[..., s:s+3] / XYZ_SCALE + XYZ_CENTER

def decode_block_quat(obs, block_idx=0):
    s = 19 + block_idx * 9
    return obs[..., s+3:s+7]

def decode_joint_pos(obs):
    return obs[..., 0:6]

def decode_joint_vel(obs):
    return obs[..., 6:12]

def decode_gripper_opening(obs):
    return np.clip(obs[..., 17] / GRIPPER_SCALE, 0.0, 1.0)


# ─── DATA LOADING ─────────────────────────────────────────────────────────────
def load_task_trajectory(data_dir, task_id):
    path = f"{data_dir}/task{task_id}.npz"
    raw  = np.load(path, allow_pickle=True)
    data = {k: raw[k] for k in raw.files}
    n    = len(data.get("observations", next(iter(data.values()))))
    print(f"  Loaded task{task_id}.npz  |  keys={list(data.keys())}  |  N={n}")
    return data

def extract_first_episode(data):
    obs = data.get("observations", next(iter(data.values()))).astype(np.float32)
    if "terminals" in data:
        terminals = data["terminals"].astype(bool).ravel()
        end_idx   = int(np.argmax(terminals)) + 1
        obs       = obs[:end_idx]
    return obs


# ─── ENVIRONMENT SETUP ────────────────────────────────────────────────────────
def make_cube_single_env(task_id, render_width=480, render_height=480):
    env = gym.make(
        "cube-single-v0",
        render_mode="rgb_array",
        width=render_width,
        height=render_height,
    )
    env.reset(options={"task_id": task_id})
    return env

def set_env_state_from_obs(env, obs):
    raw   = env.unwrapped
    model = raw._model
    data  = raw._data

    data.qpos[raw._arm_joint_ids] = obs[0:6]
    data.qvel[raw._arm_joint_ids] = obs[6:12]

    gripper_opening = float(np.clip(obs[17] / GRIPPER_SCALE, 0.0, 1.0))
    data.qpos[raw._gripper_opening_joint_id] = gripper_opening * 0.8

    block_pos  = obs[19:22] / XYZ_SCALE + XYZ_CENTER
    block_quat = obs[22:26]
    data.joint("object_joint_0").qpos[:3] = block_pos
    data.joint("object_joint_0").qpos[3:] = block_quat

    mujoco.mj_forward(model, data)


# ─── CAMERA PROJECTION ────────────────────────────────────────────────────────
def world_to_pixel(model, data, cam_id, world_point, width, height):
    """
    Project 3-D world coords → (px, py) for a named MuJoCo camera.
    cam_xmat columns = camera axes (X right, Y up, Z backward) in world frame.
    Camera looks in its local -Z, so depth = -c[2].
    Returns None if the point is behind the camera.
    """
    cam_pos = data.cam_xpos[cam_id].copy()
    cam_mat = data.cam_xmat[cam_id].copy().reshape(3, 3)   # cols = cam axes in world

    rel   = np.asarray(world_point, dtype=float) - cam_pos
    c     = cam_mat.T @ rel                                # world → camera frame
    depth = -c[2]
    if depth < 1e-4:
        return None

    fovy_rad = np.radians(model.cam_fovy[cam_id])
    f        = (height / 2.0) / np.tan(fovy_rad / 2.0)

    px = width  / 2.0 + f * c[0] / depth
    py = height / 2.0 - f * c[1] / depth   # image y is flipped vs world y
    return np.array([px, py])


# ─── RENDER FRAMES WITH TRACE OVERLAY ────────────────────────────────────────
def render_trajectory_frames(env, obs_seq, cam_name="front_pixels", trace_step=1):
    """
    Render every timestep and overlay coloured traces using PIL RGBA compositing.
      Arm  (effector): dots coloured  blue  (t=0) → red   (t=T)
      Block          : dots coloured  dark-orange → bright-green
    """
    raw   = env.unwrapped
    model = raw._model
    data  = raw._data
    W, H  = raw._render_width, raw._render_height
    N     = len(obs_seq)

    try:
        cam_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_CAMERA, cam_name)
        if cam_id < 0:
            raise ValueError
    except Exception:
        print(f"  [warn] Camera '{cam_name}' not found, falling back to 'front'.")
        cam_name = "front"
        cam_id   = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_CAMERA, "front")

    eff_world = decode_effector_pos(obs_seq)   # (N, 3)
    blk_world = decode_block_pos(obs_seq, 0)   # (N, 3)

    frames = []
    for i, obs in enumerate(obs_seq):
        set_env_state_from_obs(env, obs)
        raw_frame = raw.render(camera=cam_name)

        img  = Image.fromarray(raw_frame).convert("RGBA")
        draw = ImageDraw.Draw(img, "RGBA")

        for j in range(0, i + 1, max(1, trace_step)):
            t     = j / max(N - 1, 1)
            alpha = int(60 + 195 * t)

            px_eff = world_to_pixel(model, data, cam_id, eff_world[j], W, H)
            if px_eff is not None:
                r = int(255 * t);  b = int(255 * (1.0 - t))
                cx, cy, rd = int(px_eff[0]), int(px_eff[1]), 3
                draw.ellipse([cx-rd, cy-rd, cx+rd, cy+rd], fill=(r, 0, b, alpha))

            px_blk = world_to_pixel(model, data, cam_id, blk_world[j], W, H)
            if px_blk is not None:
                rr = int(220 * (1.0 - t));  gg = int(200 * t)
                cx, cy, rd = int(px_blk[0]), int(px_blk[1]), 4
                draw.ellipse([cx-rd, cy-rd, cx+rd, cy+rd], fill=(rr, gg, 30, alpha))

        frames.append(np.array(img.convert("RGB")))

    return frames


# ─── 2-D BIRD'S-EYE PROJECTION ───────────────────────────────────────────────
def plot_2d_projection(obs_seq, task_id, ax=None, show_legend=True):
    """
    Top-down XY view with:
      • Time-gradient polylines  (solid=arm, dashed=block, blue→red=time)
      • Red filled square    – block initial position (from task definition)
      • Green dashed square  – goal position          (from task definition)
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))

    task    = CUBE_SINGLE_TASKS[task_id]
    N       = len(obs_seq)
    eff_pos = decode_effector_pos(obs_seq)
    blk_pos = decode_block_pos(obs_seq, 0)

    for i in range(N - 1):
        t = i / max(N - 2, 1)
        ax.plot(eff_pos[i:i+2, 0], eff_pos[i:i+2, 1],
                color=(t, 0.0, 1.0 - t, 0.85), linewidth=1.8, solid_capstyle="round")
        ax.plot(blk_pos[i:i+2, 0], blk_pos[i:i+2, 1],
                color=(0.9 - 0.85*t, 0.1 + 0.85*t, 0.0, 0.80),
                linewidth=2.4, linestyle="--", solid_capstyle="round")

    ax.scatter(eff_pos[0,  0], eff_pos[0,  1], c="blue",   s=70, zorder=6, marker="o", label="Arm start")
    ax.scatter(eff_pos[-1, 0], eff_pos[-1, 1], c="red",    s=70, zorder=6, marker="o", label="Arm end")
    ax.scatter(blk_pos[0,  0], blk_pos[0,  1], c="orange", s=80, zorder=6, marker="s", label="Block start")
    ax.scatter(blk_pos[-1, 0], blk_pos[-1, 1], c="green",  s=80, zorder=6, marker="s", label="Block end")

    half = CUBE_SIZE_M / 2.0

    ix, iy = task["init_xyz"][:2]
    ax.add_patch(mpatches.Rectangle(
        (ix - half, iy - half), CUBE_SIZE_M, CUBE_SIZE_M,
        linewidth=2.2, edgecolor="crimson",
        facecolor=(1.0, 0.3, 0.3, 0.40), label="Block init (task)",
    ))

    gx, gy = task["goal_xyz"][:2]
    ax.add_patch(mpatches.Rectangle(
        (gx - half, gy - half), CUBE_SIZE_M, CUBE_SIZE_M,
        linewidth=2.2, edgecolor="limegreen", linestyle="--",
        facecolor=(0.3, 1.0, 0.3, 0.30), label="Goal (task)",
    ))

    ax.set_xlim(OBJ_X_RANGE[0] - 0.04, OBJ_X_RANGE[1] + 0.04)
    ax.set_ylim(OBJ_Y_RANGE[0] - 0.04, OBJ_Y_RANGE[1] + 0.04)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("X  [m]", fontsize=11)
    ax.set_ylabel("Y  [m]", fontsize=11)
    ax.set_title(
        f"Task {task_id}  —  {task['name']}\nsolid=arm · dashed=block · blue→red=time",
        fontsize=10,
    )
    ax.grid(True, alpha=0.22)
    if show_legend:
        ax.legend(fontsize=7.5, loc="upper right", ncol=2)
    return ax


# ─── MAIN: ANIMATED VISUALISATION ────────────────────────────────────────────
def visualize_cube_single_task(
    data_dir, task_id,
    render_width=480, render_height=480,
    fps=15, max_frames=None, trace_step=1,
    cam_name="front_pixels",
):
    """
    Two-panel animation:
      Left  – 3-D rendered view with colour-coded trajectory traces overlaid.
      Right – 2-D top-down XY projection with ticking current-position dot.
    Returns an IPython HTML animation for display() in a notebook cell.
    """
    print(f"\n{'─'*60}\n  Cube-Single Task {task_id} visualisation\n{'─'*60}")

    traj    = load_task_trajectory(data_dir, task_id)
    obs_seq = extract_first_episode(traj)

    if max_frames is not None and len(obs_seq) > max_frames:
        idx     = np.linspace(0, len(obs_seq) - 1, max_frames, dtype=int)
        obs_seq = obs_seq[idx]
    print(f"  Trajectory: {len(obs_seq)} steps  |  obs_dim: {obs_seq.shape[1]}")

    print(f"  Creating cube-single-v0 (task={task_id}) …")
    env = make_cube_single_env(task_id, render_width=render_width, render_height=render_height)

    print(f"  Rendering {len(obs_seq)} frames …")
    frames = render_trajectory_frames(env, obs_seq, cam_name=cam_name, trace_step=trace_step)
    env.close()
    print(f"  Done. Building animation …")

    fig    = plt.figure(figsize=(13, 6.5))
    ax_vid = fig.add_subplot(1, 2, 1)
    ax_2d  = fig.add_subplot(1, 2, 2)

    plot_2d_projection(obs_seq, task_id, ax=ax_2d, show_legend=True)

    eff_pos = decode_effector_pos(obs_seq)
    blk_pos = decode_block_pos(obs_seq, 0)

    (dot_arm,) = ax_2d.plot([], [], "o", color="deepskyblue", ms=10, zorder=10, label="Arm now")
    (dot_blk,) = ax_2d.plot([], [], "s", color="limegreen",   ms=10, zorder=10, label="Block now")
    time_lbl   = ax_2d.text(0.02, 0.98, "", transform=ax_2d.transAxes, va="top", ha="left", fontsize=9)
    ax_2d.legend(fontsize=7.5, loc="upper right", ncol=2)

    im = ax_vid.imshow(frames[0])
    ax_vid.axis("off")
    ax_vid.set_title(
        f"Task {task_id} – 3-D view  (● blue→red = arm, ● orange→green = block)",
        fontsize=9.5,
    )

    def _update(i):
        im.set_array(frames[i])
        dot_arm.set_data([eff_pos[i, 0]], [eff_pos[i, 1]])
        dot_blk.set_data([blk_pos[i, 0]], [blk_pos[i, 1]])
        time_lbl.set_text(f"t = {i}/{len(obs_seq)-1}")
        return [im, dot_arm, dot_blk, time_lbl]

    anim = animation.FuncAnimation(
        fig, _update, frames=len(frames),
        interval=max(1, int(1000 / fps)), blit=True,
    )
    plt.tight_layout()
    plt.close(fig)
    return HTML(anim.to_jshtml(fps=fps))


# ─── QUICK STATIC OVERVIEW ────────────────────────────────────────────────────
def plot_all_tasks_2d_summary(data_dir):
    """Static 1×5 figure: 2-D XY projection for all 5 tasks. No rendering needed."""
    fig, axes = plt.subplots(1, 5, figsize=(22, 5), constrained_layout=True)
    for tid in range(1, 6):
        traj    = load_task_trajectory(data_dir, tid)
        obs_seq = extract_first_episode(traj)
        plot_2d_projection(obs_seq, tid, ax=axes[tid - 1], show_legend=(tid == 1))
    fig.suptitle("Cube-Single Expert Trajectories – 2-D XY Projection (all 5 tasks)",
                 fontsize=13, y=1.02)
    plt.show()


# ═══════════════════════════════════════════════════════════════════════════════
#  USAGE
# ═══════════════════════════════════════════════════════════════════════════════
DATA_DIR = "../notebooks/toy_data/cube_single"  # adjust if needed

# Quick static overview (no rendering):
# plot_all_tasks_2d_summary(DATA_DIR)

# Full animated visualisation:
html = visualize_cube_single_task(DATA_DIR, task_id=1, max_frames=100, fps=12)
display(html)

# All tasks:
# for tid in range(1, 6):
#     display(visualize_cube_single_task(DATA_DIR, task_id=tid, max_frames=120, fps=12))



────────────────────────────────────────────────────────────
  Cube-Single Task 1 visualisation
────────────────────────────────────────────────────────────
  Loaded task1.npz  |  keys=['observations', 'actions', 'terminals', 'qpos', 'qvel']  |  N=64
  Trajectory: 64 steps  |  obs_dim: 28
  Creating cube-single-v0 (task=1) …


: 